# Group 2 — Dolly 1K Length Curriculum (Multi-Seed)

Curated source-only copy of the executed research notebook.

- Methods: `easy_to_hard`, `hard_to_easy`
- Seeds: `13`, `21`, `42`
- Difficulty proxy: full tokenized instruction+context length before training-time truncation
- Verified metrics: `results/group2/`

Outputs are intentionally stripped; the raw Drive notebook remains the executed research record.


In [ ]:
import torch
import numpy as np
import random
import json
import statistics
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
)
from peft import LoraConfig, get_peft_model, TaskType
from rouge_score import rouge_scorer

In [ ]:
# =========================================================
# Cell 2: Fixed config (locked from Group 1 validation)
# =========================================================
MODEL_NAME = "google/flan-t5-small"
DATASET_NAME = "databricks/databricks-dolly-15k"
SUBSET_SIZE = 1000
MAX_INPUT_LEN = 256
MAX_TARGET_LEN = 128

PER_DEVICE_TRAIN_BS = 3
GRAD_ACCUM_STEPS = 2
PER_DEVICE_EVAL_BS = 2
MAX_STEPS = 300
LEARNING_RATE = 3e-3

# Difficulty proxy: length of instruction+context (confirmed consistent with Group 1's
# embedding-input choice - instruction+context, not instruction-only).
LENGTH_USES_INSTRUCTION_ONLY = False

RESULTS_LOG_PATH = "./group2_results.json"

# =========================================================
# Cell 3: Load and format data (run once per dataset size)
# =========================================================
print("Loading dataset...")
raw_dataset = load_dataset(DATASET_NAME, split="train")
raw_dataset = raw_dataset.shuffle(seed=42).select(range(SUBSET_SIZE))

def format_example(example):
    if example.get("context"):
        prompt = f"Instruction: {example['instruction']}\nContext: {example['context']}"
    else:
        prompt = f"Instruction: {example['instruction']}"
    return {"input_text": prompt, "target_text": example["response"]}

raw_dataset = raw_dataset.map(format_example)

split = raw_dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = split["train"]
eval_dataset = split["test"]

print(f"Train size: {len(train_dataset)}, Eval size: {len(eval_dataset)}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def preprocess(example):
    model_inputs = tokenizer(
        example["input_text"], max_length=MAX_INPUT_LEN, truncation=True, padding="max_length",
    )
    labels = tokenizer(
        text_target=example["target_text"], max_length=MAX_TARGET_LEN, truncation=True, padding="max_length",
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

train_tokenized = train_dataset.map(preprocess, remove_columns=train_dataset.column_names)
eval_tokenized = eval_dataset.map(preprocess, remove_columns=eval_dataset.column_names)

In [ ]:
# =========================================================
# Cell 4: Compute difficulty (length) for every training example
# =========================================================
# Length is the full tokenizer output length of instruction+context,
# before the later training-time MAX_INPUT_LEN truncation.

def get_length_text(example):
    if LENGTH_USES_INSTRUCTION_ONLY:
        return example["instruction"]
    return example["input_text"]  # already instruction+context combined

print("Computing example lengths...")
lengths = []
for i in range(len(train_dataset)):
    text = get_length_text(train_dataset[i])
    token_count = len(tokenizer(text)["input_ids"])
    lengths.append(token_count)

# Indices sorted by length, ascending (shortest/easiest first)
easy_to_hard_indices = sorted(range(len(train_dataset)), key=lambda i: lengths[i])
hard_to_easy_indices = list(reversed(easy_to_hard_indices))

print(f"Length range: min={min(lengths)}, max={max(lengths)}, "
      f"median={sorted(lengths)[len(lengths)//2]}")

In [ ]:
# =========================================================
# Cell 5: Batch order construction (fully sorted - no shuffling within the ordering)
# =========================================================

def build_length_order(direction, n_batches):
    """Fully sorted: walks straight through the length-sorted index list, wrapping
    around (cycling) if n_batches * PER_DEVICE_TRAIN_BS exceeds the dataset size."""
    base_indices = easy_to_hard_indices if direction == "easy_to_hard" else hard_to_easy_indices
    total_needed = n_batches * PER_DEVICE_TRAIN_BS
    order = []
    while len(order) < total_needed:
        order.extend(base_indices)
    return order[:total_needed]

# =========================================================
# Cell 6: Custom sampler + Trainer subclass (same as Group 1)
# =========================================================

class FixedOrderSampler(torch.utils.data.Sampler):
    def __init__(self, indices):
        self.indices = indices
    def __iter__(self):
        return iter(self.indices)
    def __len__(self):
        return len(self.indices)

class OrderedTrainer(Seq2SeqTrainer):
    def __init__(self, *args, fixed_order=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.fixed_order = fixed_order

    def get_train_dataloader(self):
        sampler = FixedOrderSampler(self.fixed_order)
        return torch.utils.data.DataLoader(
            self.train_dataset,
            batch_size=self.args.per_device_train_batch_size,
            sampler=sampler,
            collate_fn=self.data_collator,
            drop_last=True,
        )

# =========================================================
# Cell 7: Single-run function
# =========================================================

def run_single_experiment(direction, seed):
    print(f"\n{'='*60}\nMETHOD={direction}  SEED={seed}\n{'='*60}")

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    # Fresh model load - required every run
    model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
    lora_config = LoraConfig(
        task_type=TaskType.SEQ_2_SEQ_LM, r=8, lora_alpha=16, lora_dropout=0.05,
        target_modules=["q", "v"],
    )
    model = get_peft_model(model, lora_config)

    n_batches = MAX_STEPS * GRAD_ACCUM_STEPS
    order = build_length_order(direction, n_batches)

    data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

    training_args = Seq2SeqTrainingArguments(
        output_dir=f"./output_{direction}_{seed}",
        per_device_train_batch_size=PER_DEVICE_TRAIN_BS,
        gradient_accumulation_steps=GRAD_ACCUM_STEPS,
        per_device_eval_batch_size=PER_DEVICE_EVAL_BS,
        max_steps=MAX_STEPS,
        learning_rate=LEARNING_RATE,
        logging_steps=50,
        eval_strategy="no",
        save_strategy="no",
        seed=seed,
        report_to="none",
        predict_with_generate=True,
        fp16=False,
    )

    trainer = OrderedTrainer(
        model=model,
        args=training_args,
        train_dataset=train_tokenized,
        eval_dataset=eval_tokenized,
        data_collator=data_collator,
        processing_class=tokenizer,
        fixed_order=order,
    )

    trainer.train()
    eval_results = trainer.evaluate()
    eval_loss = eval_results.get("eval_loss")
    print(f"RESULT  method={direction}  seed={seed}  eval_loss={eval_loss}")

    # --- Generation + ROUGE ---
    print("Generating outputs for ROUGE...")
    model.eval()
    predictions = []
    references = [eval_dataset[i]["target_text"] for i in range(len(eval_dataset))]
    device = "cuda" if torch.cuda.is_available() else "cpu"

    GEN_BATCH_SIZE = 64  # matched Group 1 generation-evaluation setting

    with torch.no_grad():
        for start in range(0, len(eval_dataset), GEN_BATCH_SIZE):
            end = min(start + GEN_BATCH_SIZE, len(eval_dataset))
            batch_input_ids = torch.tensor(
                [eval_tokenized[i]["input_ids"] for i in range(start, end)]
            ).to(device)
            batch_attention_mask = torch.tensor(
                [eval_tokenized[i]["attention_mask"] for i in range(start, end)]
            ).to(device)
            generated = model.generate(
                input_ids=batch_input_ids,
                attention_mask=batch_attention_mask,
                max_new_tokens=MAX_TARGET_LEN,
            )
            batch_preds = tokenizer.batch_decode(generated, skip_special_tokens=True)
            predictions.extend(batch_preds)

    scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
    rouge1_scores, rouge2_scores, rougeL_scores = [], [], []
    for pred, ref in zip(predictions, references):
        scores = scorer.score(ref, pred)
        rouge1_scores.append(scores["rouge1"].fmeasure)
        rouge2_scores.append(scores["rouge2"].fmeasure)
        rougeL_scores.append(scores["rougeL"].fmeasure)

    rouge1 = sum(rouge1_scores) / len(rouge1_scores)
    rouge2 = sum(rouge2_scores) / len(rouge2_scores)
    rougeL = sum(rougeL_scores) / len(rougeL_scores)

    print(f"RESULT  method={direction}  seed={seed}  "
          f"ROUGE-1={rouge1:.4f}  ROUGE-2={rouge2:.4f}  ROUGE-L={rougeL:.4f}")

    metrics = {
        "eval_loss": eval_loss,
        "rouge1": rouge1,
        "rouge2": rouge2,
        "rougeL": rougeL,
    }

    del model, trainer
    torch.cuda.empty_cache()

    return metrics

In [ ]:
# =========================================================
# Cell 8: Run all 6 experiments (2 methods x 3 seeds) for this dataset size
# =========================================================

METHODS = ["easy_to_hard", "hard_to_easy"]
SEEDS = [13, 21, 42]

results = []

for direction in METHODS:
    for seed in SEEDS:
        metrics = run_single_experiment(direction, seed)
        results.append({"method": direction, "seed": seed, **metrics})
        with open(RESULTS_LOG_PATH, "w") as f:
            json.dump(results, f, indent=2)

print("\n\nALL GROUP 2 RUNS COMPLETE (this dataset size)")
for r in results:
    print(r)

In [ ]:
# =========================================================
# Cell 9: Aggregate mean/std per method
# =========================================================

METRIC_KEYS = ["eval_loss", "rouge1", "rouge2", "rougeL"]

summary = {}
for direction in METHODS:
    method_runs = [r for r in results if r["method"] == direction]
    summary[direction] = {}
    for key in METRIC_KEYS:
        values = [r[key] for r in method_runs]
        summary[direction][f"mean_{key}"] = statistics.mean(values)
        summary[direction][f"std_{key}"] = statistics.stdev(values) if len(values) > 1 else 0.0
    summary[direction]["runs"] = method_runs

print(json.dumps(summary, indent=2))